In [0]:
%run ./utils/masking_functions

✔ MapleBank utils loaded: mask_sin, mask_account, postal_to_fsa, mask_email, validate_transit, validate_province, add_audit_columns


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                                DoubleType, DateType, TimestampType)
from datetime import date

VOL    = "/Volumes/workspace/default/maplebank"
SOURCE = VOL                       # where the raw CSVs sit
BRONZE = f"{VOL}/bronze"
SILVER = f"{VOL}/silver"
GOLD   = f"{VOL}/gold"

# ADF-style parameter, with a standalone default
try:
    dbutils.widgets.text("business_date", "")
    BUSINESS_DATE = dbutils.widgets.get("business_date") or str(date.today())
except:
    BUSINESS_DATE = str(date.today())

print(f"MapleBank EOD pipeline · business_date = {BUSINESS_DATE}")

MapleBank EOD pipeline · business_date = 2026-06-11


In [0]:
customer_schema = StructType([
    StructField("customer_id",    StringType(), False),
    StructField("first_name",     StringType(), True),
    StructField("last_name",      StringType(), True),
    StructField("sin",            StringType(), True),
    StructField("date_of_birth",  DateType(),   True),
    StructField("email",          StringType(), True),
    StructField("phone",          StringType(), True),
    StructField("street_address", StringType(), True),
    StructField("city",           StringType(), True),
    StructField("province",       StringType(), True),
    StructField("postal_code",    StringType(), True),
    StructField("customer_since", DateType(),   True),
])

account_schema = StructType([
    StructField("account_id",         StringType(), False),
    StructField("customer_id",        StringType(), False),
    StructField("account_number",     StringType(), True),
    StructField("account_type",       StringType(), True),
    StructField("transit_number",     StringType(), True),
    StructField("institution_number", StringType(), True),
    StructField("open_date",          DateType(),   True),
    StructField("balance_cad",        DoubleType(), True),
    StructField("branch_id",          StringType(), True),
])

branch_schema = StructType([
    StructField("branch_id",      StringType(), False),
    StructField("branch_name",    StringType(), True),
    StructField("city",           StringType(), True),
    StructField("province",       StringType(), True),
    StructField("transit_number", StringType(), True),
])

txn_schema = StructType([
    StructField("transaction_id",        StringType(),    False),
    StructField("customer_id",           StringType(),    False),
    StructField("account_id",            StringType(),    False),
    StructField("branch_id",             StringType(),    True),
    StructField("transaction_date",      DateType(),      False),
    StructField("transaction_timestamp", TimestampType(), False),
    StructField("amount_cad",            DoubleType(),    False),
    StructField("transaction_type",      StringType(),    False),
    StructField("merchant_name",         StringType(),    True),
    StructField("channel",               StringType(),    True),
])
print("Schemas defined ✔")

Schemas defined ✔


In [0]:
df_raw_customer = spark.read.option("header", True).schema(customer_schema).csv(f"{SOURCE}/dim_customer.csv")
df_raw_account  = spark.read.option("header", True).schema(account_schema).csv(f"{SOURCE}/dim_account.csv")
df_raw_branch   = spark.read.option("header", True).schema(branch_schema).csv(f"{SOURCE}/dim_branch.csv")
df_raw_txn      = spark.read.option("header", True).schema(txn_schema).csv(f"{SOURCE}/fact_transactions.csv")

# add_audit_columns comes from the utils notebook — one source of truth
df_b_customer = add_audit_columns(df_raw_customer, "dim_customer.csv",      BUSINESS_DATE)
df_b_account  = add_audit_columns(df_raw_account,  "dim_account.csv",       BUSINESS_DATE)
df_b_branch   = add_audit_columns(df_raw_branch,   "dim_branch.csv",        BUSINESS_DATE)
df_b_txn      = add_audit_columns(df_raw_txn,      "fact_transactions.csv", BUSINESS_DATE)

df_b_customer.write.format("delta").mode("overwrite").save(f"{BRONZE}/customers")
df_b_account .write.format("delta").mode("overwrite").save(f"{BRONZE}/accounts")
df_b_branch  .write.format("delta").mode("overwrite").save(f"{BRONZE}/branches")
df_b_txn     .write.format("delta").mode("overwrite") \
             .partitionBy("transaction_date").save(f"{BRONZE}/transactions")

print(f"✅ BRONZE  customers={df_b_customer.count():,}  accounts={df_b_account.count():,}  "
      f"branches={df_b_branch.count()}  transactions={df_b_txn.count():,}")

✅ BRONZE  customers=1,000  accounts=2,025  branches=15  transactions=10,000


In [0]:
df_s_customer = (
    spark.read.format("delta").load(f"{BRONZE}/customers")
    .dropDuplicates(["customer_id"])
    .filter(F.col("customer_id").isNotNull())
    .withColumn("sin_masked",      mask_sin("sin"))
    .withColumn("postal_code_fsa", postal_to_fsa("postal_code"))
    .withColumn("email_masked",    mask_email("email"))
    .withColumn("province",        F.upper(F.trim(F.col("province"))))
    .withColumn("province_valid",  validate_province("province"))
    .withColumn("age_years",
        F.floor(F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25))
    .withColumn("age_segment",
        F.when(F.col("age_years") < 25, "YOUNG_ADULT")
         .when(F.col("age_years") < 40, "MILLENNIAL")
         .when(F.col("age_years") < 60, "MID_LIFE")
         .otherwise("SENIOR"))
    .drop("sin", "email", "phone", "street_address", "postal_code")   # raw PII GONE
    .withColumn("_silver_timestamp", F.current_timestamp())
)

df_s_customer.write.format("delta").mode("overwrite").save(f"{SILVER}/customers")
print(f"✅ Silver customers: {df_s_customer.count():,}")
df_s_customer.select("customer_id","sin_masked","postal_code_fsa","email_masked","age_segment").show(3)

✅ Silver customers: 1,000
+-----------+-----------+---------------+---------------+-----------+
|customer_id| sin_masked|postal_code_fsa|   email_masked|age_segment|
+-----------+-----------+---------------+---------------+-----------+
| CUST100014|***-***--31|            Y5A|s***@example.ca|     SENIOR|
| CUST100066|***-***--69|            J9V|k***@example.ca| MILLENNIAL|
| CUST100070|***-***--52|            N6C|s***@example.ca| MILLENNIAL|
+-----------+-----------+---------------+---------------+-----------+
only showing top 3 rows


In [0]:
df_s_account = (
    spark.read.format("delta").load(f"{BRONZE}/accounts")
    .dropDuplicates(["account_id"])
    .filter(F.col("account_id").isNotNull() & F.col("customer_id").isNotNull())
    .withColumn("account_number_masked", mask_account("account_number"))
    .withColumn("transit_number_valid",  validate_transit("transit_number"))
    .drop("account_number")
    .withColumn("_silver_timestamp", F.current_timestamp())
)
df_s_account.write.format("delta").mode("overwrite").save(f"{SILVER}/accounts")

df_s_branch = (
    spark.read.format("delta").load(f"{BRONZE}/branches")
    .dropDuplicates(["branch_id"])
    .withColumn("province", F.upper(F.trim(F.col("province"))))
    .withColumn("_silver_timestamp", F.current_timestamp())
)
df_s_branch.write.format("delta").mode("overwrite").save(f"{SILVER}/branches")

print(f"✅ Silver accounts: {df_s_account.count():,}   branches: {df_s_branch.count()}")

✅ Silver accounts: 2,025   branches: 15


In [0]:
df_b_txn_r = spark.read.format("delta").load(f"{BRONZE}/transactions")

df_s_txn = (
    df_b_txn_r
    .dropDuplicates(["transaction_id"])
    .filter(F.col("transaction_id").isNotNull()
            & F.col("customer_id").isNotNull()
            & F.col("amount_cad").isNotNull()
            & (F.col("amount_cad") > 0))
    .withColumn("amount_cad", F.round(F.col("amount_cad"), 2))
    .withColumn("fintrac_flag",
        F.when(F.col("amount_cad") >= 10000, "LCTR_REPORTABLE")
         .when(F.col("amount_cad") >= 9000,  "STRUCTURING_RISK")
         .otherwise("NORMAL"))
    .withColumn("merchant_name", F.coalesce(F.col("merchant_name"), F.lit("UNKNOWN")))
    .withColumn("_silver_timestamp", F.current_timestamp())
)

df_s_txn.write.format("delta").mode("overwrite") \
    .partitionBy("transaction_date").save(f"{SILVER}/transactions")

# ── Quality report — auditors and ops expect this every run ──
b, s = df_b_txn_r.count(), df_s_txn.count()
print(f"✅ Silver transactions: {s:,}")
print(f"── Quality Report ─────────────────────")
print(f"   Bronze:   {b:>7,}")
print(f"   Silver:   {s:>7,}")
print(f"   Dropped:  {b-s:>7,}  ({(b-s)/b*100:.2f}%)")
df_s_txn.groupBy("fintrac_flag").count().show()

✅ Silver transactions: 10,000
── Quality Report ─────────────────────
   Bronze:    10,000
   Silver:    10,000
   Dropped:        0  (0.00%)
+---------------+-----+
|   fintrac_flag|count|
+---------------+-----+
|         NORMAL| 9697|
|LCTR_REPORTABLE|  303|
+---------------+-----+



## Cell 8 — GOLD: Customer 360:

In [0]:
df_txn_agg = df_s_txn.groupBy("customer_id").agg(
    F.count("*").alias("total_txn_count"),
    F.round(F.sum("amount_cad"), 2).alias("total_txn_value_cad"),
    F.round(F.avg("amount_cad"), 2).alias("avg_txn_value_cad"),
    F.max("transaction_timestamp").alias("last_txn_timestamp"),
    F.sum(F.when(F.col("fintrac_flag") == "LCTR_REPORTABLE", 1).otherwise(0)).alias("lctr_count"),
)

df_acct_agg = df_s_account.groupBy("customer_id").agg(
    F.count("*").alias("total_accounts"),
    F.round(F.sum("balance_cad"), 2).alias("total_balance_cad"),
    F.collect_set("account_type").alias("product_list"),
)

df_360 = (
    df_s_customer
    .join(df_txn_agg,  "customer_id", "left")
    .join(df_acct_agg, "customer_id", "left")
    .withColumn("customer_value_segment",
        F.when(F.col("total_txn_value_cad") >= 100000, "PLATINUM")
         .when(F.col("total_txn_value_cad") >= 50000,  "GOLD")
         .when(F.col("total_txn_value_cad") >= 10000,  "SILVER")
         .otherwise("STANDARD"))
    .withColumn("_gold_timestamp", F.current_timestamp())
)

df_360.write.format("delta").mode("overwrite").save(f"{GOLD}/customer_360")
print(f"✅ customer_360: {df_360.count():,} rows (one per customer)")
df_360.select("customer_id","province","age_segment","total_txn_count",
              "total_txn_value_cad","customer_value_segment","product_list").show(5, truncate=False)

✅ customer_360: 1,000 rows (one per customer)
+-----------+--------+-----------+---------------+-------------------+----------------------+-----------------------+
|customer_id|province|age_segment|total_txn_count|total_txn_value_cad|customer_value_segment|product_list           |
+-----------+--------+-----------+---------------+-------------------+----------------------+-----------------------+
|CUST100014 |BC      |SENIOR     |9              |17072.73           |SILVER                |[RRSP]                 |
|CUST100066 |ON      |MILLENNIAL |8              |54754.69           |GOLD                  |[CHEQUING]             |
|CUST100070 |MB      |MILLENNIAL |10             |26908.87           |SILVER                |[CHEQUING, CREDIT_CARD]|
|CUST100074 |BC      |SENIOR     |3              |5379.58            |STANDARD              |[CREDIT_CARD]          |
|CUST100107 |AB      |MILLENNIAL |9              |14712.38           |SILVER                |[RRSP, TFSA]           |
+---------

### ## Cell 9 — GOLD: daily branch summary + FINTRAC LCTR report:

In [0]:
# Daily branch summary (broadcast the tiny dim — Step 4 lesson)
df_daily_branch = (
    df_s_txn.join(F.broadcast(df_s_branch), "branch_id", "left")
    .groupBy("branch_id", "branch_name", "city", "province", "transaction_date")
    .agg(F.count("*").alias("txn_count"),
         F.round(F.sum("amount_cad"), 2).alias("total_amount_cad"),
         F.countDistinct("customer_id").alias("unique_customers"))
    .withColumn("_gold_timestamp", F.current_timestamp())
)
df_daily_branch.write.format("delta").mode("overwrite") \
    .partitionBy("transaction_date").save(f"{GOLD}/daily_branch_summary")

# FINTRAC LCTR — masked PII only (Silver guarantees this by construction)
df_lctr = (
    df_s_txn.filter(F.col("fintrac_flag") == "LCTR_REPORTABLE")
    .join(df_s_customer.select("customer_id","first_name","last_name",
                               "sin_masked","city","province","postal_code_fsa"),
          "customer_id", "left")
    .select("transaction_id","transaction_timestamp","amount_cad","transaction_type",
            "customer_id",
            F.concat_ws(" ", "first_name", "last_name").alias("customer_name"),
            "sin_masked","city","province","postal_code_fsa","branch_id",
            F.lit(BUSINESS_DATE).alias("report_date"),
            F.lit("LCTR").alias("report_type"))
)
df_lctr.write.format("delta").mode("overwrite").save(f"{GOLD}/fintrac_lctr_report")

print(f"✅ daily_branch_summary: {df_daily_branch.count():,} rows")
print(f"✅ fintrac_lctr_report:  {df_lctr.count():,} reportable transactions")
df_lctr.show(5, truncate=False)

✅ daily_branch_summary: 450 rows
✅ fintrac_lctr_report:  303 reportable transactions
+--------------+---------------------+----------+----------------+-----------+----------------+-----------+-------+--------+---------------+----------+-----------+-----------+
|transaction_id|transaction_timestamp|amount_cad|transaction_type|customer_id|customer_name   |sin_masked |city   |province|postal_code_fsa|branch_id |report_date|report_type|
+--------------+---------------------+----------+----------------+-----------+----------------+-----------+-------+--------+---------------+----------+-----------+-----------+
|TXN1003326    |2025-11-13 06:14:00  |34015.89  |ATM_WITHDRAWAL  |CUST100609 |Krystal Williams|***-***--53|Halifax|NS      |H1B            |BR_QUE_007|2026-06-11 |LCTR       |
|TXN1009112    |2025-11-11 21:23:00  |45450.58  |POS_PURCHASE    |CUST100055 |Natalie Deleon  |***-***--44|Toronto|ON      |Y3C            |BR_MON_006|2026-06-11 |LCTR       |
|TXN1001218    |2025-11-30 22:32:00

### Cell 10 — Run summary + ADF handshake:

In [0]:
print("═"*50)
print(f"  MapleBank EOD · {BUSINESS_DATE} · Run Summary")
print("═"*50)
for layer, base in [("Bronze", BRONZE), ("Silver", SILVER), ("Gold", GOLD)]:
    for t in ["customers","accounts","branches","transactions",
              "customer_360","daily_branch_summary","fintrac_lctr_report"]:
        try:
            n = spark.read.format("delta").load(f"{base}/{t}").count()
            print(f"  {layer:<7} {t:<24} {n:>8,}")
        except: pass

dbutils.notebook.exit(f"SUCCESS: business_date={BUSINESS_DATE}")